# Coding Agent


In [ ]:
# ---------------------------------------------------------------------------
# Imports: Core AutoGen components and utilities
# ---------------------------------------------------------------------------
import autogen
from autogen import AssistantAgent, UserProxyAgent
from autogen.coding import LocalCommandLineCodeExecutor
from pathlib import Path
from IPython.display import Image, Markdown
from dotenv import load_dotenv
import os

In [ ]:
load_dotenv()  # Loads .env from the current directory (or parent directories)

In [ ]:
# Step 2a: Load model configuration from CONFIG_LIST.json using the modern LLMConfig API.
# Keep this file out of source control; use .env or env vars for the actual API key.
llm_config_obj = autogen.LLMConfig.from_json(path="CONFIG_LIST.json")
config_list = llm_config_obj.config_list

# Optional: Filter config_list by model name
# Example: Filter to use only "llama-3.3-70b-versatile"
# config_list = [config for config in config_list if config.get("model") == "llama-3.3-70b-versatile"]

# Or filter to use only "openai/gpt-oss-120b"
config_list = [config for config in config_list if config.get("model") == "openai/gpt-oss-120b"]

# Or filter by partial model name match
# config_list = [config for config in config_list if "llama" in config.get("model", "").lower()]

In [ ]:
# Create a directory for code execution
work_dir = Path("coding_workspace")
work_dir.mkdir(exist_ok=True)

# Define the code executor
code_executor = LocalCommandLineCodeExecutor(
    work_dir=work_dir,  # Where files are saved
    timeout=120         # Execution timeout in seconds
)

In [ ]:
# Agent that writes code
coder = AssistantAgent(
    name="python_expert",
    llm_config={"config_list": config_list},
    system_message="""You are a helpful AI assistant specialized in Python programming.
    When asked to solve a problem, write Python code to solve it.
    Wrap code in markdown blocks like ```python ... ```.
    The user will execute the code and report the result.
    If there is an error, fix the code and output the full corrected version.
    When the task is done, reply with 'TERMINATE'."""
)

In [ ]:
# Agent that executes code
user_proxy = UserProxyAgent(
    name="executor_user",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=10,
    is_termination_msg=lambda x: x.get("content", "").strip().endswith("TERMINATE"),
    code_execution_config={"executor": code_executor}
)

In [ ]:
# Task description
task = """
Plot a chart comparing Tesla (TSLA) and Nvidia (NVDA) stock prices for the last 6 months.
Save the plot as 'stock_comparison.png'.
Make sure to install yfinance and matplotlib if needed.
"""

user_proxy.initiate_chat(coder, message=task)

In [ ]:
# ---------------------------------------------------------------------------
# Display the chart produced by the agent (saved in the "coding" work directory)
# ---------------------------------------------------------------------------
Image("coding_workspace/stock_comparison.png")

### Research report generation

The same assistant + user proxy can handle more involved tasks. In this example, the agent is asked to fetch papers (e.g. from arXiv), summarize them, and write a single Markdown report. The conversation may take several turns (install deps, run scripts, retry on errors). When running in a notebook you may need to press Enter to allow auto-reply or type `exit` to stop.

In [ ]:
# Wrapper to send a natural-language task to the assistant. The user proxy
# starts the chat; the assistant may respond with code; the proxy executes it
# and sends results back until the task is done (or you type 'exit').
# Agent that writes code
coder = AssistantAgent(
    name="python_expert",
    llm_config={"config_list": config_list},
    system_message="""You are a helpful AI assistant specialized in Python programming.
    When asked to solve a problem, write Python code to solve it.
    Wrap code in markdown blocks like ```python ... ```.
    The user will execute the code and report the result.
    If there is an error, fix the code and output the full corrected version.
    When the task is done, reply with 'TERMINATE'."""
)
# Agent that executes code
user_proxy = UserProxyAgent(
    name="executor_user",
    human_input_mode="TERMINATE",
    max_consecutive_auto_reply=5,
    # is_termination_msg=lambda x: x.get("content", "").strip().endswith("TERMINATE"),
    code_execution_config={"executor": code_executor}
)

def execute_agent(prompt: str):
    """Send a task to the assistant via the user proxy."""
    return user_proxy.initiate_chat(coder, message=prompt)

# Example 1: Data + plotting. The assistant will typically suggest a Python script
# (e.g. using yfinance + matplotlib), the proxy runs it in the "coding" folder,
# and the plot is saved as stock_price_change.png there.
execute_agent(
    """Fetch 2 papers about using small language models and
    summarize them into a one single research report file named research-small-llms.md. Use arXiv API to fetch the papers and provide the final report in Markdown format."""
)